# 20 — Inference figures and release

**Objective.** Assemble frozen source tables for manuscript exhibits, generate non-decorative figures, anonymize case outputs, scan for identifiers, and package a raw-data-free reproducibility release.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

The release excludes source data, credentials, protocol signing keys, test-access logs, and named-company case lists.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("20", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import shutil, zipfile
import pandas as pd
from cruxvc.hashing import sha256_file
from cruxvc.io import read_json, write_json, write_table
from cruxvc.plotting import plot_portfolio_overlap, plot_risk_coverage, plot_specification_effects
from cruxvc.validation import release_scan

required = {
    "rq1": P.inference / "rq1_specification_effects.csv",
    "rq2": P.inference / "rq2_family_effects.csv",
    "controls": P.controls / "rq3_control_summary.csv",
    "driver_taxonomy": P.inference / "construct_robust_driver_taxonomy.csv",
    "decision": P.decision / "topk_portfolio_overlap.csv",
    "gates": P.selective / "deployable_gate_curves.csv",
    "gate_summary": P.selective / "gate_partial_aurc_summary.csv",
    "feature_robustness": P.inference / "strict_extended_feature_robustness.csv",
    "feature_explanation_sensitivity": P.inference / "strict_extended_explanation_sensitivity.csv",
    "scarcity_envelope": P.inference / "event_count_matched_scarcity_envelope.csv",
    "landmark_strata": P.inference / "landmark_strata_performance.csv",
}
missing = [str(path) for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Release cannot be assembled; missing required analysis outputs: {missing}")


In [ ]:
rq1 = pd.read_csv(required["rq1"]).rename(columns={"delta_spec": "estimate"})
decision = pd.read_csv(required["decision"])
gates = pd.read_csv(required["gates"])
figure_paths = [
    plot_specification_effects(rq1[["contrast", "estimate", "ci_lower", "ci_upper"]], P.figures / "figure_rq1_specification_effects.png"),
    plot_portfolio_overlap(decision, P.figures / "figure_portfolio_overlap.png"),
]
for loss, frame in gates.groupby("loss"):
    safe = str(loss).replace("/", "_")
    figure_paths.append(plot_risk_coverage(frame, P.figures / f"figure_risk_coverage_{safe}.png"))

In [ ]:
table_dir = P.release / "tables"
table_dir.mkdir(parents=True, exist_ok=True)
copied_tables = []
for name, source in required.items():
    target = table_dir / f"{name}{source.suffix}"
    shutil.copy2(source, target)
    copied_tables.append(target)
inventory = pd.DataFrame([
    {"artifact": name, "source": str(path.relative_to(P.root)), "release_copy": str((table_dir / f'{name}{path.suffix}').relative_to(P.root))}
    for name, path in required.items()
])
inventory_path = write_table(inventory, P.release / "release_table_inventory.csv")

In [ ]:
staging = P.release / "crux_vc_reproducibility_release"
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)
for directory in ["config", "notebooks", "src", "tools", "tests", "docs"]:
    shutil.copytree(
        P.root / directory,
        staging / directory,
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".pytest_cache"),
    )
shutil.copytree(
    P.protocol,
    staging / "protocol",
    dirs_exist_ok=True,
    ignore=shutil.ignore_patterns("test_access_log.jsonl", "*.key", ".signing_key"),
)
shutil.copytree(table_dir, staging / "results" / "tables", dirs_exist_ok=True)
(staging / "figures").mkdir(parents=True, exist_ok=True)
for figure in figure_paths:
    shutil.copy2(figure, staging / "figures" / figure.name)
for filename in ["README.md", "NOTEBOOKS.md", "VALIDATION_REPORT.md", "LICENSE", "requirements.txt", "pyproject.toml", "CITATION.cff", "Makefile"]:
    source = P.root / filename
    if source.exists():
        shutil.copy2(source, staging / filename)
problems = release_scan(staging)
if problems:
    raise RuntimeError("Release identifier scan failed: " + " | ".join(problems))


In [ ]:
release_zip = P.release / "CRUX_VC_reproducibility_release.zip"
if release_zip.exists():
    release_zip.unlink()
with zipfile.ZipFile(release_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(staging.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(staging.parent))
release_manifest = write_json({
    "release_zip": str(release_zip.relative_to(P.root)),
    "release_zip_sha256": sha256_file(release_zip),
    "raw_data_included": False,
    "named_startups_included": False,
    "figures": [str(path.relative_to(P.root)) for path in figure_paths],
    "tables": [str(path.relative_to(P.root)) for path in copied_tables],
    "identifier_scan_problems": problems,
    "esrc_summary_included": (P.selective / "esrc_summary.json").exists(),
}, P.release / "release_manifest.json")
CTX.recorder.complete([*figure_paths, *copied_tables, inventory_path, release_zip, release_manifest])
print(read_json(release_manifest))
